# Part 4: Deployment & Storage Theory

## 🎯 Learning Objectives
- Understand data storage layers (Bronze/Silver/Gold)
- Learn deployment strategies (Batch vs Real-time)
- Explore Azure architecture for production systems
- Understand MLOps best practices

## 📚 This is THEORY - No Code to Run!

This notebook explains how we would deploy our discount recommendation system in a real production environment at Albert Heijn.

## Part A: Data Storage Architecture

### 🥉 Bronze Layer (Raw Data)
**Purpose:** Store raw, immutable data exactly as received

**Our Example:**
```
data/raw/
├── reviews.csv       ← Customer reviews (with duplicates, missing values)
├── inventory.csv     ← Stock levels (raw sensor data)
└── sales.csv         ← Transaction data (unprocessed)
```

**Real World (Azure):**
- **Azure Blob Storage** - Hot tier for recent data
- **Retention:** Keep forever (compliance, debugging)
- **Access:** Read-only after initial write

**Why keep raw data?**
- Reprocess if ETL logic changes
- Audit trail for compliance
- Debugging data quality issues

---

### 🥈 Silver Layer (Cleaned Data)
**Purpose:** Clean, validated, deduplicated data

**Our Example:**
```
data/silver/
├── reviews_clean.csv      ← No duplicates, validated ratings, cleaned text
├── inventory_clean.csv    ← Valid stock levels, proper dates
└── sales_clean.csv        ← Validated sales records
```

**Real World (Azure):**
- **Azure Data Lake Storage Gen2** - Optimized for analytics
- **Format:** Parquet (columnar, compressed)
- **Partitioning:** By date (`/year=2025/month=01/day=09/`)
- **Retention:** 2-3 years

**Processing Tool:**
- **Azure Databricks** - Spark jobs for large-scale cleaning

---

### 🥇 Gold Layer (Business-Ready)
**Purpose:** Aggregated, enriched data ready for analytics/ML

**Our Example:**
```
data/gold/
├── final_dataset.csv                    ← Reviews + Stock + Sales merged
└── product_discount_recommendations.csv ← Product-level metrics
```

**Real World (Azure):**
- **Azure SQL Database** - For PowerBI dashboards
- **Azure Synapse Analytics** - For large analytical queries
- **Cosmos DB** - For real-time API access

**Who Uses This:**
- 📊 **PowerBI Dashboards** - Store managers see discount recommendations
- 🤖 **ML Models** - Training data for retraining
- 📱 **Mobile Apps** - Display discounted products to customers

## Data Lifecycle Management

### Retention Policies

| Layer | Retention | Storage Tier | Why? |
|-------|-----------|--------------|------|
| Bronze | Forever | Cool/Archive | Compliance, reprocessing |
| Silver | 2-3 years | Hot | Active analytics |
| Gold | 1 year | Hot/Premium | Fast access for dashboards |

### Azure Blob Storage Lifecycle Rules

```yaml
Lifecycle Policy:
  - Bronze data older than 90 days → Move to Cool tier (€0.01/GB)
  - Bronze data older than 1 year → Move to Archive tier (€0.002/GB)
  - Silver data older than 2 years → Delete
  - Gold data older than 1 year → Delete
```

**Cost Savings:**
- Hot: €0.02/GB/month
- Cool: €0.01/GB/month (50% cheaper)
- Archive: €0.002/GB/month (90% cheaper)

**Trade-off:** Archive retrieval takes hours vs seconds

## Part B: ML Model Deployment

### 🎯 Deployment Strategy: Batch vs Real-Time

#### Option 1: Batch Processing (Our Use Case)
**What:** Process all products once per night

**Schedule:**
```
Every night at 2 AM:
1. Load new reviews from last 24 hours
2. Run sentiment model on new reviews
3. Aggregate by product
4. Calculate discount recommendations
5. Update database for store systems
```

**Azure Architecture:**
```
Azure Data Factory (Orchestration)
  ↓
Databricks Notebook (ETL + ML)
  ↓
Azure SQL Database (Results)
  ↓
PowerBI Dashboard (Visualization)
  ↓
Store Systems (Apply Discounts)
```

**Pros:**
- ✅ Simple to implement
- ✅ Cost-effective
- ✅ Easy to debug
- ✅ Suitable for daily pricing updates

**Cons:**
- ❌ Not real-time
- ❌ Discounts updated only once per day

---

#### Option 2: Real-Time API (Future Enhancement)
**What:** Predict sentiment immediately when review is submitted

**Flow:**
```
Customer submits review
  ↓
API call to Azure Functions
  ↓
Load model from Azure ML
  ↓
Predict sentiment
  ↓
Update product metrics in Cosmos DB
  ↓
Recalculate discount (if threshold crossed)
```

**Azure Architecture:**
```
Azure API Management
  ↓
Azure Functions (Serverless ML inference)
  ↓
Azure ML Model Endpoint
  ↓
Cosmos DB (Real-time product metrics)
```

**Pros:**
- ✅ Instant response to customer feedback
- ✅ Dynamic pricing throughout the day

**Cons:**
- ❌ More complex
- ❌ Higher cost (API calls)
- ❌ Requires monitoring

## Model Serialization & Versioning

### Saving Models (What We Did)
```python
import pickle

# Save model
with open('sentiment_model.pkl', 'wb') as f:
    pickle.dump(model, f)

# Load model
with open('sentiment_model.pkl', 'rb') as f:
    model = pickle.load(f)
```

### Production Approach (Azure ML)
```python
from azureml.core import Model

# Register model with metadata
model = Model.register(
    workspace=ws,
    model_name='sentiment_v1',
    model_path='sentiment_model.pkl',
    description='Logistic Regression for review sentiment',
    tags={
        'accuracy': '0.87',
        'training_date': '2025-01-09',
        'data_version': 'v2.1'
    }
)
```

### Model Versioning
```
sentiment_v1.0 → 87% accuracy (deployed)
sentiment_v1.1 → 89% accuracy (testing)
sentiment_v2.0 → 91% accuracy (with BERT, in development)
```

**Benefits:**
- Roll back if new model performs worse
- A/B test different model versions
- Track performance over time

## Monitoring & Retraining

### What to Monitor

#### 1. Model Performance Drift
```python
# Track accuracy over time
if current_accuracy < 0.80:  # Below threshold
    trigger_retrain_alert()
```

**Warning Signs:**
- Accuracy drops from 87% to 75%
- More misclassifications of negative reviews
- Customer complaints about incorrect discounts

#### 2. Data Drift
```python
# Monitor input data distribution
if new_data_distribution != training_distribution:
    trigger_data_drift_alert()
```

**Examples:**
- New products added (model hasn't seen them)
- Seasonal changes (Christmas reviews different from summer)
- Language changes (new slang, abbreviations)

#### 3. Business Metrics
```python
# Track discount impact
- Food waste reduction %
- Revenue from discounted products
- Customer satisfaction scores
```

### Retraining Strategy

**Scheduled Retraining:**
```
Every 3 months:
1. Collect new reviews from last 3 months
2. Retrain model with updated data
3. Evaluate on hold-out test set
4. If accuracy > current_model: deploy
5. Else: investigate data quality issues
```

**Triggered Retraining:**
```
If performance drops below 80%:
1. Alert data science team
2. Investigate root cause
3. Collect more training data if needed
4. Retrain with augmented dataset
5. A/B test new model vs old
```

## Real-World Example: AH Dynamic Markdown

### Actual Architecture at Albert Heijn

```
┌─────────────────────────────────────────────────────────────┐
│                     DATA SOURCES                             │
├─────────────────────────────────────────────────────────────┤
│ • Store Systems (sales, inventory)                          │
│ • Customer Reviews (website, app)                           │
│ • Weather Data (affects fresh produce demand)               │
│ • Promotions Calendar                                        │
└──────────────────┬──────────────────────────────────────────┘
                   ↓
┌─────────────────────────────────────────────────────────────┐
│              BRONZE LAYER (Azure Blob Storage)               │
├─────────────────────────────────────────────────────────────┤
│ Raw data ingestion via Azure Data Factory                   │
│ ~100GB/day of transaction data                              │
└──────────────────┬──────────────────────────────────────────┘
                   ↓
┌─────────────────────────────────────────────────────────────┐
│          SILVER LAYER (Azure Data Lake Gen2)                 │
├─────────────────────────────────────────────────────────────┤
│ ETL Processing with Databricks (PySpark)                    │
│ - Data cleaning & validation                                │
│ - Deduplication                                              │
│ - Schema enforcement                                         │
└──────────────────┬──────────────────────────────────────────┘
                   ↓
┌─────────────────────────────────────────────────────────────┐
│           GOLD LAYER (Azure Synapse Analytics)               │
├─────────────────────────────────────────────────────────────┤
│ • Product-level sentiment scores                            │
│ • Discount recommendations                                   │
│ • Sales forecasts                                            │
│ • Food waste predictions                                     │
└──────────────────┬──────────────────────────────────────────┘
                   ↓
┌─────────────────────────────────────────────────────────────┐
│                    SERVING LAYER                             │
├─────────────────────────────────────────────────────────────┤
│ • PowerBI (Manager dashboards)                              │
│ • Store systems (Automatic discount application)            │
│ • Mobile app (Customer-facing discounts)                    │
└─────────────────────────────────────────────────────────────┘
```

### Key Components

1. **Azure Data Factory** - Orchestrates ETL pipelines
2. **Azure Databricks** - Scales to process millions of reviews
3. **Azure ML** - Hosts and monitors ML models
4. **Azure Synapse** - Analytics warehouse for BI queries
5. **PowerBI** - Dashboards for business users

### Scale
- **1000+ stores** across Netherlands
- **Millions of transactions** per day
- **Hundreds of thousands of reviews** per month
- **~20,000 products** in catalog

### Business Impact
- 📉 **15-20% reduction** in food waste
- 💰 **€50M annual savings** through optimized discounting
- 😊 **5-point increase** in customer satisfaction scores
- 🌍 **Sustainability** - Less waste, better for environment

## MLOps Best Practices

### 1. Version Everything
```
Git Repository:
├── code/              (model training scripts)
├── configs/           (hyperparameters)
├── data_versions/     (dataset fingerprints)
└── models/            (model artifacts)
```

### 2. Automate Testing
```python
# Unit test for prediction function
def test_predict_sentiment():
    assert predict_sentiment('worst product ever') == 'negative'
    assert predict_sentiment('love it!') == 'positive'
    
# Integration test
def test_end_to_end_pipeline():
    # Test entire workflow
    raw_data → clean → train → predict → validate
```

### 3. CI/CD Pipeline
```yaml
On code commit:
  1. Run unit tests
  2. Train model on sample data
  3. Validate accuracy > threshold
  4. Deploy to staging environment
  5. Run integration tests
  6. Deploy to production (if all pass)
```

### 4. Monitoring Dashboard
```
Real-time metrics:
- Model accuracy (hourly)
- Prediction latency (ms)
- API uptime (%)
- Cost per prediction (€)
- Data quality alerts
```

### 5. Incident Response
```
If model accuracy < 80%:
  → Auto-alert: data science team
  → Fallback: Use rule-based discounts
  → Investigate: Data drift? Bug? Infrastructure?
  → Fix: Retrain or rollback to previous version
```

## Cost Estimation (Azure)

### Monthly Costs for Our System

| Service | Usage | Cost/Month |
|---------|-------|------------|
| Azure Blob Storage (Bronze) | 1TB | €20 |
| Azure Data Lake (Silver/Gold) | 500GB | €10 |
| Azure Databricks (ETL) | 100 hours/month | €400 |
| Azure ML (Model hosting) | Standard tier | €200 |
| Azure SQL Database | Business Critical | €600 |
| Azure Data Factory (Pipelines) | 50 pipelines | €50 |
| PowerBI Premium | Per user | €200 |
| **TOTAL** | | **~€1,500/month** |

### ROI Calculation
```
Monthly Cost: €1,500
Food Waste Savings: €50,000,000/year = €4,166,667/month

ROI = (Savings - Cost) / Cost = 277,678%
```

**Payback Period:** Less than 1 hour 🚀

## 💡 Summary: Workshop Recap

### What We Learned

#### Part 1: Business Analysis
- ✅ Always start with business questions
- ✅ Validate data availability and quality
- ✅ Check feasibility before building

#### Part 2: ETL & Data Preparation
- ✅ Bronze → Silver → Gold architecture
- ✅ Clean datasets **separately** before merging
- ✅ Intermediate layers for debugging

#### Part 3: ML Model Training
- ✅ Train sentiment analysis model
- ✅ **Aggregate sentiment by product** (not individual reviews)
- ✅ **Multi-factor discount formula**:
  - Sentiment (50%)
  - Stock pressure (30%)
  - Sales velocity (20%)
- ✅ Business impact analysis

#### Part 4: Deployment (This Notebook)
- ✅ Data storage strategies
- ✅ Batch vs real-time deployment
- ✅ MLOps best practices
- ✅ Real-world Azure architecture

### Key Takeaways for Your Career

1. **Think End-to-End:** ML is only 10% of the solution
   - 30% data engineering
   - 30% infrastructure
   - 20% monitoring
   - 10% ML modeling

2. **Production ≠ Notebook:** Real systems need:
   - Error handling
   - Monitoring
   - Scalability
   - Cost optimization

3. **Business Value First:** Technology serves business goals
   - €50M savings >> 87% vs 89% accuracy
   - Fast implementation >> perfect solution
   - Maintenance >> one-time deployment

### Next Steps (Self-Study)

1. **Explore Azure Free Tier:**
   - Create free account
   - Try Blob Storage, Databricks, ML

2. **Improve Our Model:**
   - Try different algorithms (Random Forest, BERT)
   - Add more features (product category, seasonality)
   - Experiment with discount formula weights

3. **Build Portfolio Project:**
   - Deploy this on Azure/AWS
   - Create PowerBI dashboard
   - Document on GitHub

4. **Learn MLOps Tools:**
   - MLflow (experiment tracking)
   - Kubeflow (Kubernetes ML pipelines)
   - Airflow (workflow orchestration)

## 🎉 Congratulations!

You've completed a realistic data science project from:
- Business analysis → ETL → ML → Deployment theory

This mirrors real-world work at companies like Albert Heijn, where:
- Data engineering is critical
- Business context drives decisions  
- Production deployment requires planning

**Questions? Discussion? Let's talk!** 💬